# LAB 7 – Comparación de SRI con Precisión R

Se compara la **Precisión R** de dos sistemas sobre las consultas de CACM, con métricas calculadas a Z = 100 documentos:

- **A**: nuestro sistema (`Practica6/CACM_metricas_Z100.txt`)
- **B**: sistema del equipo vecino (`Practica7/CACM_metricas_Z100_otro_equipo.txt`)

Ambos archivos tienen columnas `Consulta, Precisión, Recuerdo, Medida F, Precisión R`; solo se usa la última.

In [1]:
import os
import numpy as np

In [2]:
ruta_A = os.path.join("..", "Practica6", "CACM_metricas_Z100.txt")
ruta_B = os.path.join("..", "Practica7", "CACM_metricas_Z100_otro_equipo.txt")
ruta_salida = os.path.join("..", "Practica7", "CACM_comparacion_PR.txt")

Se lee la Precisión R (última columna) de cada consulta. Los números de consulta se convierten a entero.

In [3]:
# devuelve {consulta: precision_r}
def leer_precision_r(ruta):
    pr = {}
    with open(ruta, encoding="utf-8") as f:
        next(f)  # encabezado
        for linea in f:
            campos = linea.split("\t")
            if len(campos) >= 5:
                pr[int(campos[0])] = float(campos[-1])
    return pr

PR_A = leer_precision_r(ruta_A)
PR_B = leer_precision_r(ruta_B)
print("Consultas en A:", len(PR_A))
print("Consultas en B:", len(PR_B))

Consultas en A: 52
Consultas en B: 64


VALIDACIONES

El equipo vecino incluye consultas que no tienen juicios de relevancia en `qrels.text` (todas sus métricas valen 0).
Esas consultas no se pueden evaluar (|R| = 0), así que solo se comparan las consultas que aparecen en ambos archivos.

In [4]:
consultas = sorted(set(PR_A) & set(PR_B))
print("Consultas comparadas:", len(consultas))
print("Solo en A:", sorted(set(PR_A) - set(PR_B)))
print("Solo en B (sin juicios de relevancia):", sorted(set(PR_B) - set(PR_A)))
print("Valores de B en esas consultas:", [PR_B[q] for q in sorted(set(PR_B) - set(PR_A))])

Consultas comparadas: 52
Solo en A: []
Solo en B (sin juicios de relevancia): [34, 35, 41, 46, 47, 50, 51, 52, 53, 54, 55, 56]
Valores de B en esas consultas: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


TABLA DE COMPARACIÓN

Para cada consulta i: PR_A(i), PR_B(i) y la diferencia PR_A(i) − PR_B(i).
Diferencia positiva → A fue mejor en esa consulta; negativa → B fue mejor.
La última fila P(PR) es el promedio de cada columna.

In [5]:
def tabla(PR_A, PR_B, consultas):
    filas = [(q, PR_A[q], PR_B[q], PR_A[q] - PR_B[q]) for q in consultas]
    prom = np.mean([f[1:] for f in filas], axis=0)
    return filas, prom

filas, prom = tabla(PR_A, PR_B, consultas)
print("%-7s %-9s %-9s %-14s" % ("Q", "PR_A(i)", "PR_B(i)", "PR_A(i)-PR_B(i)"))
for q, a, b, d in filas:
    print("%-7d %-9.4f %-9.4f %+.4f" % (q, a, b, d))
print("%-7s %-9.4f %-9.4f %+.4f" % ("P(PR)", *prom))

Q       PR_A(i)   PR_B(i)   PR_A(i)-PR_B(i)
1       0.2000    0.2000    +0.0000
2       0.0000    0.0000    +0.0000
3       0.1667    0.1667    +0.0000
4       0.1667    0.1667    +0.0000
5       0.1250    0.0000    +0.1250
6       0.0000    0.0000    +0.0000
7       0.3214    0.3214    +0.0000
8       0.0000    0.0000    +0.0000
9       0.1111    0.2222    -0.1111
10      0.6286    0.5429    +0.0857
11      0.4211    0.4211    +0.0000
12      0.4000    0.4000    +0.0000
13      0.3636    0.2727    +0.0909
14      0.4773    0.5000    -0.0227
15      0.2000    0.2000    +0.0000
16      0.1176    0.1176    +0.0000
17      0.1875    0.1250    +0.0625
18      0.1818    0.1818    +0.0000
19      0.4545    0.3636    +0.0909
20      0.0000    0.0000    +0.0000
21      0.0909    0.0000    +0.0909
22      0.4706    0.4706    +0.0000
23      0.0000    0.0000    +0.0000
24      0.1538    0.1538    +0.0000
25      0.2745    0.3137    -0.0392
26      0.4333    0.4667    -0.0334
27      0.3793    0.

In [6]:
def guardar_tabla(filas, prom, ruta):
    with open(ruta, "w", encoding="utf-8") as salida:
        salida.write("Q\tPR_A(i)\tPR_B(i)\tPR_A(i)-PR_B(i)\n")
        for q, a, b, d in filas:
            salida.write(f"{q}\t{a:.4f}\t{b:.4f}\t{d:.4f}\n")
        salida.write("P(PR)\t" + "\t".join(f"{v:.4f}" for v in prom) + "\n")
    print("Archivo generado:", ruta)

guardar_tabla(filas, prom, ruta_salida)

Archivo generado: ..\Practica7\CACM_comparacion_PR.txt


ANÁLISIS DE LAS DIFERENCIAS

Se cuenta en cuántas consultas gana cada sistema y cuáles consultas pesan más en el promedio.

In [7]:
dif = np.array([d for _, _, _, d in filas])
print("A mejor que B:", int(np.sum(dif > 1e-9)), "consultas")
print("B mejor que A:", int(np.sum(dif < -1e-9)), "consultas")
print("Empate:       ", int(np.sum(np.abs(dif) <= 1e-9)), "consultas")
print()
print("Consultas con mayor diferencia:")
for q, a, b, d in sorted(filas, key=lambda f: abs(f[3]), reverse=True)[:5]:
    print("  Q%-3d PR_A = %.4f  PR_B = %.4f  diferencia = %+.4f" % (q, a, b, d))

A mejor que B: 10 consultas
B mejor que A: 11 consultas
Empate:        31 consultas

Consultas con mayor diferencia:
  Q64  PR_A = 0.0000  PR_B = 1.0000  diferencia = -1.0000
  Q31  PR_A = 0.5000  PR_B = 1.0000  diferencia = -0.5000
  Q5   PR_A = 0.1250  PR_B = 0.0000  diferencia = +0.1250
  Q49  PR_A = 0.1250  PR_B = 0.0000  diferencia = +0.1250
  Q9   PR_A = 0.1111  PR_B = 0.2222  diferencia = -0.1111


La diferencia del promedio se concentra en pocas consultas con muy pocos relevantes.
Con |R| = 1 o 2 la Precisión R solo puede valer 0, 0.5 o 1, así que una sola consulta mueve mucho el promedio.
Se revisa cómo cambia P(PR) si se quitan las consultas 64 y 31.

In [8]:
for excluir in [[64], [64, 31]]:
    resto = [q for q in consultas if q not in excluir]
    _, p = tabla(PR_A, PR_B, resto)
    print("Sin", excluir, "-> P(PR)_A = %.4f  P(PR)_B = %.4f  diferencia = %+.4f" % tuple(p))

Sin [64] -> P(PR)_A = 0.2669  P(PR)_B = 0.2693  diferencia = -0.0024
Sin [64, 31] -> P(PR)_A = 0.2623  P(PR)_B = 0.2547  diferencia = +0.0076


Consulta 64: tiene un solo documento relevante (2651). Se busca en el ranking de nuestro sistema (Práctica 5).

In [9]:
ruta_ranking = os.path.join("..", "Practica5", "CACM_tf_idf_rels.txt")
docs_64 = []
with open(ruta_ranking, encoding="utf-8") as f:
    for linea in f:
        campos = linea.split()
        if len(campos) == 3 and int(campos[0]) == 64:
            docs_64.append(int(campos[1]))
print("Documentos recuperados por A para Q64:", len(docs_64))
print("¿Aparece el documento 2651?", 2651 in docs_64)

Documentos recuperados por A para Q64: 610
¿Aparece el documento 2651? False


## Respuestas

**¿Cómo se usa la Precisión R para comparar SRI?**

La Precisión R de una consulta es la precisión en la posición |R|, donde |R| es el número de documentos relevantes de esa consulta:
se revisan los primeros |R| documentos del ranking y se divide el número de relevantes encontrados entre |R|.
Vale 1 si el sistema pone todos los relevantes en los primeros |R| lugares. Como el corte se adapta a cada consulta, las consultas con
muchos o pocos relevantes se miden en la misma escala.

Para comparar dos sistemas A y B se calcula PR_A(i) y PR_B(i) en cada consulta i y la diferencia PR_A(i) − PR_B(i):
si es positiva A fue mejor en esa consulta, si es negativa B fue mejor. El promedio P(PR) de cada sistema resume el desempeño global,
y el número de consultas ganadas por cada uno muestra si la ventaja es general o depende de unas pocas consultas.

**¿Qué sistema fue mejor?**

Con las 52 consultas que tienen juicios de relevancia: P(PR)_A = 0.2618 y P(PR)_B = 0.2834, diferencia −0.0216.
**El sistema B (equipo vecino) fue ligeramente mejor.** A ganó en 10 consultas, B en 11 y hubo empate en 31.

**¿Por qué nuestro sistema fue peor?**

- Los dos sistemas son casi iguales (ambos tf-idf con coseno): empatan en 31 de 52 consultas, y en las demás las diferencias son pequeñas y se reparten casi por igual.
- La ventaja de B sale casi toda de dos consultas con muy pocos relevantes:
  - **Q64** (|R| = 1): nuestro sistema nunca recuperó el documento 2651 (no comparte términos con la consulta después de nuestro preprocesamiento), así que PR = 0; B lo puso en primer lugar, PR = 1. Diferencia −1.0.
  - **Q31** (|R| = 2): nosotros pusimos uno de los dos relevantes en las 2 primeras posiciones (PR = 0.5); B puso ambos (PR = 1).
- Sin Q64 la diferencia es prácticamente cero (−0.0024), y sin Q64 y Q31 nuestro sistema queda arriba (+0.0076). La diferencia se explica sobre todo por el preprocesamiento (tokenización, stopwords, stemming), que hizo que nuestro sistema perdiera el único relevante de Q64.
- Nuestro sistema fue mejor en consultas como Q5, Q49, Q48, Q21 y Q10, donde B tuvo PR más bajo.